# Testing the Relationships Between Measures of Variation
Using the full multiverse analysis on the SIAB dataset, we would like to confirm mathematical identities we have derived.

## Setup

In [ ]:
# for development purposes
%load_ext autoreload
%autoreload 2

In [ ]:
# imports
import numpy as np
import pandas as pd
from pathlib import Path
from src.metrics.metrics import (
    organize_multiverse_results,
    between_bucket_variation,
    total_variance_decomposition,
)
import matplotlib.pyplot as plt

In [ ]:
# get all csv files in the data directory for run of interest
data_dir = Path("output/runs/1/data")
csv_files = list(data_dir.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files.")

# read all files with mixed types
arrays = []
for i, f in enumerate(csv_files):
    arr = np.genfromtxt(f, delimiter=',', dtype=None, encoding='utf-8')
    arrays.append(arr)
    if (i + 1) % 1000 == 0:
        print(f"Loaded {i+1}/{len(csv_files)} files")

# stack all arrays vertically
data = np.concatenate(arrays)
# convert data to pandas df
df = pd.DataFrame(data)
# remove all rows that are exact duplicates, since they are removing the rows of column names
df = df.drop_duplicates()
# make first row the header
df.columns = df.iloc[0]
df = df[1:].reset_index(drop=True)

Inspect data to make sure it has been read properly.

In [ ]:
df.head()

## Organize Results by Dimension & Output Metric
Before we break down the variation induced by each judgment call in the multiverse, we need to organize the multiverse outputs in a way that makes them easy to analyze.

In [ ]:
# set up inputs for `organize_multiverse_results`
dimension_order = ["spell_gap_threshold",
 "new_job_notif_threshold",
 "new_job_no_notif_threshold",
 "missing_strategy",
 "split_seed",
 "model"]
# prepend mv_dim_ to all dimension names
dimension_order = ["mv_dim_" + dim for dim in dimension_order]
performance_metrics = ["acc", "f1", "roc_auc"]
# prepend test_ to all performance metric names
performance_metrics = ["test_" + metric for metric in performance_metrics]

# make all cols that start with test_ numeric
for col in df.columns:
    if col.startswith("test_"):
        df[col] = pd.to_numeric(df[col])

In [ ]:
# call helper function to make a structured df
df = organize_multiverse_results(df, dimension_order, performance_metrics)

Inspect the structured multiverse output.

In [ ]:
# look at data
df.head()

In [ ]:
# ensure types are correct
df.dtypes

In [ ]:
variance_decomposition = total_variance_decomposition(df, "test_acc")

In [ ]:
variance_decomposition

In [ ]:
# look at variance decomposition results
for dimension, decomposition in variance_decomposition.items():
    print(f"Dimension: {dimension}")
    print(f"Total Variance: {decomposition['var_total']}")
    print(f"Variance of Group Means: {decomposition['var_of_expected_values_across_groups']}")
    print(f"Expected Value of Group Variances: {decomposition['expected_var_within_groups']}")
    print(f"Difference: {decomposition['var_total'] - (decomposition['var_of_expected_values_across_groups'] + decomposition['expected_var_within_groups'])}")
    print("-" * 50)

In [ ]:
between = between_bucket_variation(df, "test_acc")

In [ ]:
between

In [ ]:
# define all decision dimensions
decision_dims = {
    "mv_dim_missing_strategy": "Missing Strategy",
    "mv_dim_model": "Model",
    "mv_dim_new_job_no_notif_threshold": "New Job No-Notification Threshold",
    "mv_dim_new_job_notif_threshold": "New Job Notification Threshold",
    "mv_dim_spell_gap_threshold": "Spell Gap Threshold",
    "mv_dim_split_seed": "Train-Test Split Seed",
}

In [ ]:
# plot scatterplot confirming that "between" is equal expected within-group var
plt.figure(figsize=(10, 6))
for dim in dimension_order:
    plt.scatter(variance_decomposition[dim]["expected_var_within_groups"], between[dim], label=decision_dims[dim], s=100)
plt.xlabel("Within-Bucket Variation", fontsize=12)
plt.ylabel("Between-Bucket Variation", fontsize=12)
plt.title("Within vs Between Bucket Variation for Each Dimension", fontsize=14, fontweight='bold')
# add a line for y=x to confirm that points are on the line
max_val = max(max(variance_decomposition[dim]["expected_var_within_groups"] for dim in dimension_order), max(between[dim] for dim in dimension_order))
plt.plot([0, max_val], [0, max_val], color='red', linestyle='--', label='y=x')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# confirm that they are equal
for dim in dimension_order:
    assert np.isclose(variance_decomposition[dim]["expected_var_within_groups"], between[dim]), f"Variance decomposition does not hold for dimension {dim}"